## VLM Shellgame

- VLM is **molmo2** (has a Object Tracker model as video enc)  
(Theoretically should outperform the glm model -- OPENROUTER: **allenai/molmo-2-8b**)
- Videos: A,B,C labeled on the table with 3 inverted cups. Each video starts by showing where the puck is then covers it with a cup.  
then cups are moved around in shell game style.
- evaluated on single videos of shell game in POV.
- 3 types of videos: slow, medium, fast to see temporal resoning of the model
- For each video model picks from A,B,C


In [3]:
## conda env vgd has openai
import base64
import os
import json
import time
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd
import numpy as np
from tqdm import tqdm

# import cv2

In [4]:
# Setting the max characters to be displayed
pd.set_option('display.max_colwidth', None)

# display full dataframe pandas / output window widened
pd.set_option('display.width', 10000) # None

# Setting the precision for float columns
pd.set_option('display.precision', 2)

# Preparing main index_df with vqa videos

In [5]:
## index_df
index_df = pd.read_csv('/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/index.csv')
print(index_df.shape)

index_df = index_df[index_df['session'].str.startswith('shellgame')].reset_index(drop=True)
print(index_df.shape)

index_df['cam'] = 'cam0'
index_df = index_df.rename(columns={'id':'video_id'})

print(index_df.shape)
index_df.head(2)

(228, 4)
(32, 4)
(32, 5)


,video_id,template,placeholders,session,cam
0,261117-155127,C,PUCK,shellgame-medium,cam0
1,261117-155207,B,PUCK,shellgame-medium,cam0


In [6]:
labels_list = index_df.template.unique().tolist()
labels_list.sort()

index_df.template.value_counts()

template
C    11
A    11
B    10
Name: count, dtype: int64

In [7]:
index_df.session.value_counts()

session
shellgame-slow      12
shellgame-medium    10
shellgame-fast      10
Name: count, dtype: int64

### select videos

In [8]:
for i, row in index_df.iterrows():
    print(f"'{row['video_id']}', # --> {row['template']:5} -- {row['session']} ")


'261117-155127', # --> C     -- shellgame-medium 
'261117-155207', # --> B     -- shellgame-medium 
'261117-155252', # --> A     -- shellgame-medium 
'261117-155426', # --> C     -- shellgame-medium 
'261117-155503', # --> C     -- shellgame-medium 
'261117-155543', # --> B     -- shellgame-fast 
'261117-155621', # --> C     -- shellgame-fast 
'261117-155722', # --> B     -- shellgame-fast 
'261117-160815', # --> A     -- shellgame-fast 
'261117-160858', # --> A     -- shellgame-fast 
'261117-160936', # --> B     -- shellgame-fast 
'261117-161017', # --> C     -- shellgame-fast 
'261117-161053', # --> B     -- shellgame-fast 
'261117-161121', # --> C     -- shellgame-fast 
'261117-161150', # --> C     -- shellgame-fast 
'261117-161257', # --> B     -- shellgame-medium 
'261117-161345', # --> A     -- shellgame-medium 
'261117-161420', # --> A     -- shellgame-medium 
'261117-161455', # --> B     -- shellgame-medium 
'261117-161544', # --> A     -- shellgame-medium 
'261117-161811', # -

In [9]:
vids_list = [
    '261117-155127', # --> C     -- shellgame-medium 
    '261117-155207', # --> B     -- shellgame-medium 
    '261117-155252', # --> A     -- shellgame-medium 
    '261117-155426', # --> C     -- shellgame-medium 
    '261117-155503', # --> C     -- shellgame-medium 
    '261117-155543', # --> B     -- shellgame-fast 
    '261117-155621', # --> C     -- shellgame-fast 
    '261117-155722', # --> B     -- shellgame-fast 
    '261117-160815', # --> A     -- shellgame-fast 
    '261117-160858', # --> A     -- shellgame-fast 
    '261117-160936', # --> B     -- shellgame-fast 
    '261117-161017', # --> C     -- shellgame-fast 
    '261117-161053', # --> B     -- shellgame-fast 
    '261117-161121', # --> C     -- shellgame-fast 
    '261117-161150', # --> C     -- shellgame-fast 
    '261117-161257', # --> B     -- shellgame-medium 
    '261117-161345', # --> A     -- shellgame-medium 
    '261117-161420', # --> A     -- shellgame-medium 
    '261117-161455', # --> B     -- shellgame-medium 
    '261117-161544', # --> A     -- shellgame-medium 
    '261117-161811', # --> C     -- shellgame-slow 
    '261117-161859', # --> A     -- shellgame-slow 
    '261117-161936', # --> A     -- shellgame-slow 
    '261117-162009', # --> C     -- shellgame-slow 
    '261117-162259', # --> A     -- shellgame-slow 
    '261117-162319', # --> C     -- shellgame-slow 
    '261117-162343', # --> B     -- shellgame-slow 
    '261117-162402', # --> B     -- shellgame-slow 
    '261117-162434', # --> A     -- shellgame-slow 
    '261117-162457', # --> A     -- shellgame-slow 
    '261117-162515', # --> B     -- shellgame-slow 
    '261117-162538', # --> C     -- shellgame-slow 

]
print(len(vids_list))

32


In [10]:
print(index_df.shape)
index_df = index_df[index_df.video_id.isin(vids_list)]
print(index_df.shape)

(32, 5)
(32, 5)


In [13]:
import shutil
# ------------- chunk to copy videos ------------- #
source_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/"
dest_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/zz_temp_dump/"

# 2. Clear destination directory if it exists, then recreate it
if os.path.exists(dest_dir):
    shutil.rmtree(dest_dir)
os.makedirs(dest_dir, exist_ok=True)

# 3. Get unique video IDs and copy matching files
unique_video_ids = vids_list

for filename in os.listdir(source_dir):
    if any(video_id in filename for video_id in unique_video_ids):
        shutil.copy(os.path.join(source_dir, filename), os.path.join(dest_dir, filename))
# ------------------------------------------------ #

# VLM Evaluation 

### Compress Videos. 9second videos too long and high def. glm-5.3-Flash can handle <10MB vids and mine are well over 15mb per single view

## compress to lower res and store as mp4

In [46]:
import os
import subprocess

raw_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/raw"
compressed_dir = "data/compressed_videos"

os.makedirs(compressed_dir, exist_ok=True)

for vid in vids_list:
    # Strip extension if vids_list already includes '.avi'
    stem = os.path.splitext(vid)[0]
    
    input_path = os.path.join(raw_dir, f"{stem}_cam0.avi")       # Change to f"{stem}_cam0.avi" if suffix is still used
    output_path = os.path.join(compressed_dir, f"{stem}_cam0.mp4")

    if os.path.exists(output_path):
        print(f"[-] Skipping (already exists): {stem}_cam0.mp4")
        continue

    if not os.path.exists(input_path):
        print(f"[!] Warning (missing input): {input_path}")
        continue

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-loglevel", "error",
            "-i", input_path,
            "-vf", "fps=20,scale=-2:480p",
            "-c:v", "libx264",
            "-crf", "23",
            "-an",
            output_path,
        ],
        check=True,
    )

    orig_mb = os.path.getsize(input_path) / (1024 * 1024)
    new_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"[*] Converted {stem}: {orig_mb:.2f} MB -> {new_mb:.2f} MB")

[-] Skipping (already exists): 261117-155127_cam0.mp4
[-] Skipping (already exists): 261117-155207_cam0.mp4
[*] Converted 261117-155252: 15.34 MB -> 2.31 MB
[-] Skipping (already exists): 261117-155426_cam0.mp4
[-] Skipping (already exists): 261117-155503_cam0.mp4
[*] Converted 261117-155543: 10.72 MB -> 1.71 MB
[*] Converted 261117-155621: 15.05 MB -> 2.45 MB
[*] Converted 261117-155722: 14.99 MB -> 2.41 MB
[*] Converted 261117-160815: 11.08 MB -> 1.75 MB
[*] Converted 261117-160858: 15.47 MB -> 2.41 MB
[*] Converted 261117-160936: 12.56 MB -> 1.97 MB
[*] Converted 261117-161017: 15.04 MB -> 2.35 MB
[*] Converted 261117-161053: 11.51 MB -> 1.75 MB
[*] Converted 261117-161121: 11.00 MB -> 1.67 MB
[*] Converted 261117-161150: 17.47 MB -> 2.64 MB
[*] Converted 261117-161257: 14.33 MB -> 1.99 MB
[*] Converted 261117-161345: 15.82 MB -> 2.31 MB
[*] Converted 261117-161420: 14.69 MB -> 2.13 MB
[*] Converted 261117-161455: 15.23 MB -> 2.09 MB
[*] Converted 261117-161544: 16.18 MB -> 2.29 MB


In [ ]:
## 6mins total to compress all 32 videos to 480p 20fps mp4 format.

### Run one video

In [33]:
# Load environment variables from .env
load_dotenv()

True

In [34]:
client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"), # openrouter change
    base_url="https://openrouter.ai/api/v1", # openrouter change
    )
client

In [54]:
# Encode combined video to base64
video_name = vids_list[1]

video_file_path = f'data/compressed_videos/{video_name}_cam0.mp4'

with open(video_file_path, "rb") as f:
    b64_video = base64.b64encode(f.read()).decode("utf-8")
    b64_size_mb = len(b64_video) / (1024 ** 2)
    print(f"Base64 payload size: {b64_size_mb:.2f} MB")

Base64 payload size: 2.92 MB


In [75]:
len(b64_video)

3060820

In [74]:
assert not b64_video.startswith("data:"), "Duplicate data: prefix detected"
assert "\n" not in b64_video, "Newlines found in base64 string"

In [55]:
system_prompt = (
    "You are an expert visual object-tracking and spatial-temporal reasoning model. "
    "You are provided with a single bird's-eye view video of a table with three clearly marked positions: A, B, and C, each having an inverted cup. "
    "At the beginning of the video, a puck is revealed under one of the cups, after which the cups are shuffled. "
    "Track the target cup continuously across all movements to determine its final position."
)

user_prompt = f"""
Analyze this bird's-eye view video of a shell game and determine which position (A, B, or C) currently hides the puck at the end of the video.

Valid Positions: {{"A", "B", "C"}}

Instructions:
1. Identify the starting position (A, B, or C) where the puck is initially revealed under a cup at the start of the video.
2. Track the cup with the puck chronologically through every move or swap from start to finish, noting how the target cup's position changes at each step.
3. State your final predicted position ("A", "B", or "C") where the puck is located at the end of the video.
4. Provide a confidence score as a float between 0.0 and 1.0 denoting your certainty in the final predicted position (where 1.0 is completely certain and 0.33 is random guessing).

Output strictly in JSON matching this schema:|
{{
  "initial_position": "<A, B, or C>",
  "swap_sequence": "A -> C -> A -> B",
  "predicted_position": "<A, B, or C>",
  "confidence": 0.0
}}
"""

print(user_prompt)


Analyze this bird's-eye view video of a shell game and determine which position (A, B, or C) currently hides the puck at the end of the video.

Valid Positions: {"A", "B", "C"}

Instructions:
1. Identify the starting position (A, B, or C) where the puck is initially revealed under a cup at the start of the video.
2. Track the cup with the puck chronologically through every move or swap from start to finish, noting how the target cup's position changes at each step.
3. State your final predicted position ("A", "B", or "C") where the puck is located at the end of the video.
4. Provide a confidence score as a float between 0.0 and 1.0 denoting your certainty in the final predicted position (where 1.0 is completely certain and 0.33 is random guessing).

Output strictly in JSON matching this schema:|
{
  "initial_position": "<A, B, or C>",
  "swap_sequence": "A -> C -> A -> B",
  "predicted_position": "<A, B, or C>",
  "confidence": 0.0
}



In [81]:
start_time = time.time()

response = client.chat.completions.create(
    model="google/gemini-3.8-flash",
    messages=[
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": [
                {
                    "type": "file",
                    "file": {
                        "filename": "video.mp4",
                        "file_data": f"data:video/mp4;base64,{b64_video}",
                    },
                },
                {
                    "type": "text",
                    "text": user_prompt,
                },
            ],
        },
    ],
    response_format={"type": "json_object"},
    temperature=0.1,
)

print("Prompt tokens:", response.usage.prompt_tokens)

elapsed_time = round(time.time() - start_time, 2)

print(f"Elapsed time: {elapsed_time}s")

Prompt tokens: 879
Elapsed time: 40.72s


In [86]:
import time
import requests

start_time = time.time()

res = requests.post(
    "https://openrouter.ai/api/v1/responses",
    headers={
        "Authorization": f"Bearer {os.getenv('OPENROUTER_API_KEY')}",
        "Content-Type": "application/json",
    },
    json={
        "model": "google/gemini-3.8-flash",
        "instructions": system_prompt,
        "input": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_video",
                        "video_url": f"data:video/mp4;base64,{b64_video}",
                    },
                    {
                        "type": "input_text",
                        "text": user_prompt,
                    },
                ],
            }
        ],
        "text": {"format": {"type": "json_object"}},
        "temperature": 0.1,
    },
    timeout=120,
)

res.raise_for_status()
data = res.json()
elapsed_time = round(time.time() - start_time, 2)

print("Input tokens:", data["usage"]["input_tokens"])
print("Output:", data["output"][0]["content"][0]["text"])

Input tokens: 879
Output: **Tracking puck location**

I'm focused on precisely identifying the puck's position throughout the video sequence. Initial analysis confirms it began under the cup at position C. My current efforts are dedicated to following its movement from frame to frame.

**Analyzing cup movements**

I'm scrutinizing the precise movements of the cups, particularly how the cup initially covering the puck at C is handled. It appears the right hand may be manipulating the cup that was at C, possibly lifting it or swapping it with cup B. I am trying to track its exact trajectory.

**Tracking cup positions**

I'm refining my understanding of cup movements. It appears the right hand, initially over B/C, might be moving a cup downwards at 00:03, while the left hand holds cup A. At 00:04, there are three cups, with the left hand holding one at C/B.

**Discerning cup locations**

I'm piecing together the cup positions. At 00:01, cup C is missing. The right hand seems to be lifting

In [89]:
import os
import requests

models = requests.get("https://openrouter.ai/api/v1/models").json()["data"]
for m in models:
    if "gemini" in m["id"]:
        print(m["id"], "->", m.get("architecture", {}).get("input_modalities"))

# Also inspect your previous response object `data`:
print("Actual model:", data.get("model"))
print("Actual provider:", data.get("provider"))

google/gemini-4-neon -> ['audio', 'file', 'image', 'text', 'video']
google/gemini-4-neon:batch -> ['audio', 'file', 'image', 'text', 'video']
google/gemini-4-hydrogen -> ['audio', 'file', 'image', 'text', 'video']
google/gemini-4-hydrogen:batch -> ['audio', 'file', 'image', 'text', 'video']
google/gemini-4-carbon -> ['audio', 'file', 'image', 'text', 'video']
google/gemini-4-carbon:batch -> ['audio', 'file', 'image', 'text', 'video']
google/gemini-omni-1.2-flash -> ['text', 'image', 'video']
google/gemini-omni-1.1-flash -> ['text', 'image', 'video']
google/gemini-4-argon -> ['audio', 'file', 'image', 'text', 'video']
google/gemini-4-argon:batch -> ['audio', 'file', 'image', 'text', 'video']
google/gemini-nano-banana-2.1 -> ['image', 'text']
google/gemini-3.8-flash -> ['text', 'image', 'video', 'file', 'audio']
google/gemini-3.8-flash:batch -> ['text', 'image', 'video', 'file', 'audio']
google/gemini-3.7-flash -> ['text', 'image', 'video', 'file', 'audio']
google/gemini-3.7-flash:batch 

In [88]:
data["usage"]

{'input_tokens': 879,
 'input_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0},
 'output_tokens': 5004,
 'output_tokens_details': {'reasoning_tokens': 4957},
 'total_tokens': 5883,
 'cost': 0.01942425,
 'is_byok': False,
 'cost_details': {'upstream_inference_cost': 0.01942425,
  'upstream_inference_input_cost': 0.00065925,
  'upstream_inference_output_cost': 0.018765}}

In [82]:
prediction_json = json.loads(response.choices[0].message.content)
prediction_json.keys()

dict_keys(['initial_position', 'swap_sequence', 'predicted_position', 'confidence'])

In [83]:
# Extract JSON prediction content
prediction_json = json.loads(response.choices[0].message.content)

# Extract token & performance metadata
usage = response.usage

# 3. Update the dictionary to log both video file names

## this try except is from my testing when i wasnt giving the user_prompt to the models and model was returning confidence like 'low','high'
# try: 
#     conf = float(prediction_json.get("confidence", 0.0))
# except ValueError:
#     conf = 0.0

run_result = {
    "video_file": os.path.basename(video_file_path),
    "model": response.model,
    "elapsed_seconds": elapsed_time,
    "prompt_tokens": usage.prompt_tokens if usage else None,
    "completion_tokens": usage.completion_tokens if usage else None,
    "total_tokens": usage.total_tokens if usage else None,
    "initial_position": prediction_json.get("initial_position"),
    "predicted_position": prediction_json.get("predicted_position"),
    "confidence": float(prediction_json.get("confidence", 0.0)),
    # "confidence": conf,
    "swap_sequence": prediction_json.get("swap_sequence"),
}

# Append result as a single JSON line
with open('data/vlm_eval_results/shellgame_predictions_testing_models.jsonl', "a", encoding="utf-8") as f:
    f.write(json.dumps(run_result) + "\n")

print(json.dumps(run_result, indent=2))

{
  "video_file": "261117-155207_cam0.mp4",
  "model": "google/gemini-3.8-flash",
  "elapsed_seconds": 40.72,
  "prompt_tokens": 879,
  "completion_tokens": 5350,
  "total_tokens": 6229,
  "initial_position": "C",
  "predicted_position": "C",
  "confidence": 0.72,
  "swap_sequence": "C -> B -> C"
}


In [84]:
# 1. Load jsonl file directly into a DataFrame
df_testing_models = pd.read_json("data/vlm_eval_results/shellgame_predictions_testing_models.jsonl", lines=True)

cost = df_testing_models['prompt_tokens']*0.0000003779 + df_testing_models['completion_tokens']*0.000004245
df_testing_models.insert(6, 'cost',  '$ ' +round(cost, 4).astype(str))

print(df_testing_models.shape)
print(df_testing_models)
print(f"\nTotal Cost of Testing: $ {round(sum(cost), 3)}")

(10, 11)
               video_file                          model  elapsed_seconds  prompt_tokens  completion_tokens  total_tokens      cost initial_position predicted_position  confidence          swap_sequence
0  261117-155127_cam0.mp4                   z-ai/glm-5.3            14.07            123               1859          1982  $ 0.0079              NaN                NaN        0.00                    NaN
1  261117-155127_cam0.mp4             z-ai/glm-5.3-flash             1.81            123                120           243  $ 0.0006              NaN                NaN        0.33                    NaN
2  261117-155127_cam0.mp4  google/gemini-3-flash-preview             2.44            113                 90           203  $ 0.0004              NaN                NaN        0.00                    NaN
3  261117-155127_cam0.mp4  google/gemini-3-flash-preview             2.34            878                 50           928  $ 0.0005                A                  A        0.90

## Main Experiment: For loop over many videos

In [ ]:
# Load environment variables from .env
load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"), # openrouter change
    base_url="https://openrouter.ai/api/v1", # openrouter change
    )
client

In [73]:
len(vids_list[1:])

26

In [ ]:
from tqdm import tqdm

videos_dir = 'data/compressed_videos/'

pbar = tqdm(vids_list[1:], desc="Processing Videos")

for video_name in pbar:
    # Make file path and load video in base64 format
    video_file_path = videos_dir + video_name + '_cam0.mp4'
    with open(video_file_path, "rb") as f:
        b64_video = base64.b64encode(f.read()).decode("utf-8")

    # system_prompts and user_prompt exactly same for all videos.
    
    # API call
    start_time = time.time()

    response = client.chat.completions.create(
        model="glm-5.3-flash",
        messages=[
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": user_prompt},
                    {
                        "type": "video_url",
                        "video_url": {"url": f"data:video/mp4;base64,{b64_video}"},
                    },
                ],
            },
        ],
        response_format={"type": "json_object"},
        temperature=0.1,
    )

    elapsed_time = round(time.time() - start_time, 2)

    # Extract JSON prediction content
    prediction_json = json.loads(response.choices[0].message.content)

    # Extract token & performance metadata
    usage = response.usage

    # 3. Update the dictionary to log both video file names
    run_result = {
        "video_file": os.path.basename(video_file_path),
        "model": response.model,
        "elapsed_seconds": elapsed_time,
        "prompt_tokens": usage.prompt_tokens if usage else None,
        "completion_tokens": usage.completion_tokens if usage else None,
        "total_tokens": usage.total_tokens if usage else None,
        "initial_position": prediction_json.get("initial_position"),
        "predicted_position": prediction_json.get("predicted_position"),
        "confidence": float(prediction_json.get("confidence", 0.0)),
        "swap_sequence": prediction_json.get("swap_sequence"),
    }

    # Append result as a single JSON line
    with open('data/vlm_eval_results/shellgame_predictions_qwen3.5-9b.jsonl', "a", encoding="utf-8") as f:
        f.write(json.dumps(run_result) + "\n")


Processing Videos: 100%|██████████| 26/26 [07:53<00:00, 18.20s/it]


# Loading and analyzing the jsonl outputs of the VLM
## Main Experiment: shellgame

In [10]:
# 1. Load jsonl file directly into a DataFrame
df = pd.read_json("data/vlm_eval_results/shellgame_predictions.jsonl", lines=True)

print(df.shape)
print(df.head(10))

(30, 10)
               video_file          model  elapsed_seconds  prompt_tokens  completion_tokens  total_tokens initial_position predicted_position  confidence               swap_sequence
0  261117-155207_cam0.mp4  glm-5.3-flash           479.36           9975              21434         31409                C                  B        0.70                 C -> A -> B
1  261117-155127_cam0.mp4  glm-5.3-flash          1343.85           9975               1072         11047                A                  A        1.00            A -> B -> B -> A
2  261117-155252_cam0.mp4  glm-5.3-flash           531.72          11166               2055         13221                A                  C        1.00       A -> B -> A -> C -> C
3  261117-155426_cam0.mp4  glm-5.3-flash           358.85          11166               6388         17554                B                  A        0.95            B -> A -> B -> A
4  261117-155503_cam0.mp4  glm-5.3-flash           422.55           9964         

In [11]:
index_df['video_file'] = index_df.video_id + '_cam0.mp4'
print(index_df.shape)

df = pd.merge(df, index_df, how = 'left', on = ['video_file']) # LEFT JOIN
print(df.shape)
print(df.sample(3))

(32, 6)
(30, 15)
                video_file          model  elapsed_seconds  prompt_tokens  completion_tokens  total_tokens initial_position predicted_position  confidence          swap_sequence       video_id template placeholders           session   cam
2   261117-155252_cam0.mp4  glm-5.3-flash           531.72          11166               2055         13221                A                  C         1.0  A -> B -> A -> C -> C  261117-155252        A         PUCK  shellgame-medium  cam0
16  261117-161811_cam0.mp4  glm-5.3-flash           207.52          11166               1381         12547                A                  B         1.0       A -> B -> C -> B  261117-161811        C         PUCK    shellgame-slow  cam0
19  261117-162009_cam0.mp4  glm-5.3-flash           398.51          12368               1968         14336                A                  B         1.0       A -> C -> C -> B  261117-162009        C         PUCK    shellgame-slow  cam0


In [13]:
df['correct_top1'] = df['predicted_position'] == df['template']

print(f'Examples: {df.shape[0]}')
print(f'correct: {df.correct_top1.sum():4}')
print(f'top1acc: {df.correct_top1.mean():4}')

Examples: 30
correct:   12
top1acc:  0.4


In [20]:
import textwrap
import pandas as pd

pd.set_option('display.float_format', '{:,.3f}'.format)

def analyze_df(df):
    print('=' * 20, 'DF Stats by Session', '=' * 20)

    stats = df.groupby(['session']).agg(
        videos=('video_file', 'count'),
        top1_acc=('correct_top1', 'mean'),
        avg_confidence=('confidence', 'mean'),
    )
    print(stats)
    print('\n')

    print('=' * 30, 'Individual Videos by session', '=' * 30)

    for session in df['session'].unique():
        print('\n' + '=' * 30, session, '=' * 30)

        for i, row in df[df['session'] == session].iterrows():
            # Include ground-truth column if present in your merged DataFrame
            summary_cols = [
                col for col in [
                    'video_id', 'session', 'initial_position',
                    'template', 'predicted_position', 'confidence', 'correct_top1'
                ] if col in df.columns
            ]
            print(row[summary_cols].to_frame().T.to_string(index=False))

            print('\nSwap Sequence: ', textwrap.fill(str(row['swap_sequence']), width=100))
            print('-' * 60)

In [26]:
analyze_df(df)

==================== DF Stats by Session ====================
                  videos  top1_acc  avg_confidence
session                                           
shellgame-fast         8     0.250           0.999
shellgame-medium      10     0.300           0.963
shellgame-slow        12     0.583           0.958


============================== Individual Videos by session ==============================

============================== shellgame-medium ==============================
     video_id          session initial_position template predicted_position confidence correct_top1
261117-155207 shellgame-medium                C        B                  B      0.700         True

Swap Sequence:  C -> A -> B
------------------------------------------------------------
     video_id          session initial_position template predicted_position confidence correct_top1
261117-155127 shellgame-medium                A        C                  A      1.000        False

Swap Sequence:  A -

### A little random pred experiment

In [26]:
from random import shuffle

x = [i for i in range(10)]
print(x)
shuffle(x)
print(x)


[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[1, 8, 7, 2, 3, 9, 5, 4, 0, 6]


In [24]:
import numpy as np

rng = np.random.default_rng(49)
df["random_pred"] = rng.choice(["A", "B", "C"], size=len(df))

df['correct_top1_rand'] = df['random_pred'] == df['template']

print(f'Examples: {df.shape[0]}')
print(f'correct: {df.correct_top1_rand.sum():4}')
print(f'top1acc: {df.correct_top1_rand.mean():4}')

Examples: 30
correct:    9
top1acc:  0.3


In [25]:
df.groupby(['session']).agg(
        videos=('video_file', 'count'),
        top1_acc=('correct_top1_rand', 'mean'),
        avg_confidence=('confidence', 'mean'),
    )

,videos,top1_acc,avg_confidence
session,,,
shellgame-fast,8,0.375,0.999
shellgame-medium,10,0.000,0.963
shellgame-slow,12,0.500,0.958


In [28]:
set(vids_list) - set(df.video_id)

{'261117-155543', '261117-160858', '261117-160936', '261117-161121'}

In [28]:
df.groupby(['session']).agg(
        videos=('video_file', 'count'),
        top1_acc=('correct_top1_rand', 'mean'),
        avg_confidence=('confidence', 'mean'),
        time=('elapsed_seconds', 'mean'),
        prompt_tokens = ('prompt_tokens', 'mean'),
        total_tokens=('total_tokens', 'mean')
    )

,videos,top1_acc,avg_confidence,time,prompt_tokens,total_tokens
session,,,,,,
shellgame-fast,8,0.375,0.999,173.514,"10,414.750","12,941.500"
shellgame-medium,10,0.000,0.963,380.109,"10,687.400","15,344.700"
shellgame-slow,12,0.500,0.958,141.838,"10,264.500","12,888.250"
